# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

RANDOM_STATE = 42

# Ищем датасет под тем именем, которое есть рядом с ноутбуком
possible_paths = [
    Path("auto_dataset(2).csv"),
    Path("auto_dataset.csv")
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден auto_dataset(2).csv или auto_dataset.csv"
    )

data = pd.read_csv(DATA_PATH)

print("Размер датасета:", data.shape)
print("\nТипы признаков:")
print(data.dtypes)

print("\nКоличество пропусков:")
print(data.isna().sum())

data.head()

Размер датасета: (1000, 10)

Типы признаков:
brand                  str
model                  str
vehicleType            str
gearbox                str
fuelType               str
notRepairedDamage      str
powerPS              int64
kilometer            int64
autoAgeMonths        int64
price                int64
dtype: object

Количество пропусков:
brand                0
model                0
vehicleType          0
gearbox              0
fuelType             0
notRepairedDamage    0
powerPS              0
kilometer            0
autoAgeMonths        0
price                0
dtype: int64


,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [3]:
TARGET = "price"

# Отделяем признаки и целевую переменную
X_raw = data.drop(columns=TARGET)
y_raw = data[TARGET].to_numpy(dtype=float)

# Категориальные и числовые признаки
categorical_cols = (
    X_raw
    .select_dtypes(include=["object", "string", "category"])
    .columns
    .tolist()
)

numeric_cols = [
    col
    for col in X_raw.columns
    if col not in categorical_cols
]

print("Категориальные признаки:")
print(categorical_cols)

print("\nЧисловые признаки:")
print(numeric_cols)

print("\nРазмер X:", X_raw.shape)
print("Размер y:", y_raw.shape)

Категориальные признаки:
['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']

Числовые признаки:
['powerPS', 'kilometer', 'autoAgeMonths']

Размер X: (1000, 9)
Размер y: (1000,)


3. Разбейте датасет на train val test в отношении 8:1:1

In [5]:

X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=RANDOM_STATE,
    shuffle=True
)

X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df,
    y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("Train:", X_train_df.shape)
print("Validation:", X_val_df.shape)
print("Test:", X_test_df.shape)



try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    # Для старых версий sklearn
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            encoder,
            categorical_cols
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_cols
        )
    ],
    sparse_threshold=0
)

X_train = preprocessor.fit_transform(X_train_df)
X_val = preprocessor.transform(X_val_df)
X_test = preprocessor.transform(X_test_df)

X_train = np.asarray(X_train, dtype=float)
X_val = np.asarray(X_val, dtype=float)
X_test = np.asarray(X_test, dtype=float)



X_train = np.column_stack([
    np.ones(X_train.shape[0]),
    X_train
])

X_val = np.column_stack([
    np.ones(X_val.shape[0]),
    X_val
])

X_test = np.column_stack([
    np.ones(X_test.shape[0]),
    X_test
])





print("\nПосле preprocessing:")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

Train: (800, 9)
Validation: (100, 9)
Test: (100, 9)

После preprocessing:
X_train: (800, 195)
X_val: (100, 195)
X_test: (100, 195)

y_train: (800,)
y_val: (100,)
y_test: (100,)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:

STEP_GRID = np.logspace(-5, 0, 6)

MAX_ITER = 5000

def get_eta(
    base_step,
    k,
    lr_mode,
    s0=1.0,
    p=0.5
):

    if lr_mode == "constant":
        return float(base_step)

    if lr_mode == "decay":
        return (
            float(base_step)
            * (s0 / (s0 + k)) ** p
        )

    raise ValueError(
        "lr_mode должен быть 'constant' или 'decay'"
    )


def predict_original_scale(X, w):
    # y обучается в исходном масштабе, поэтому обратное масштабирование не нужно
    return X @ w


def evaluate_weights(X, y_original, w):

    prediction = predict_original_scale(
        X,
        w
    )

    mse = float(
        np.mean(
            (y_original - prediction) ** 2
        )
    )

    r2 = float(
        r2_score(
            y_original,
            prediction
        )
    )

    return mse, r2


def train_optimizer(
    method,
    lr_mode,
    base_step,
    max_iter=MAX_ITER,
    batch_size=32,
    alpha=0.9,
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8,
    random_state=RANDOM_STATE
):


    n, d = X_train.shape

    # Начинаем с нулевых весов
    w = np.zeros(d)

    rng = np.random.default_rng(
        random_state
    )


    if method == "SAG":

        # Для каждого train-объекта храним
        # последний вычисленный градиент
        gradient_memory = np.zeros(
            (n, d)
        )

        average_gradient = np.zeros(d)


    elif method == "Momentum":

        # h_0 = 0
        h = np.zeros(d)


    elif method == "Adam":

        # m_0 = 0, v_0 = 0
        m = np.zeros(d)
        v = np.zeros(d)

    best_val_loss, _ = evaluate_weights(
        X_val,
        y_val,
        w
    )

    best_weights = w.copy()
    best_iteration = 0

    val_history = []


    for k in range(max_iter):


        if method in [
            "VGD",
            "Momentum",
            "Adam"
        ]:

            gradient = (
                2.0 / n
            ) * X_train.T @ (
                X_train @ w
                - y_train
            )

        elif method == "SGD":

            batch_indices = rng.choice(
                n,
                size=min(batch_size, n),
                replace=False
            )

            X_batch = X_train[
                batch_indices
            ]

            y_batch = y_train[
                batch_indices
            ]

            gradient = (
                2.0 / len(batch_indices)
            ) * X_batch.T @ (
                X_batch @ w
                - y_batch
            )


        elif method == "SAG":

            j = int(
                rng.integers(
                    0,
                    n
                )
            )

            x_j = X_train[j]
            y_j = y_train[j]

            old_gradient = (
                gradient_memory[j].copy()
            )

            new_gradient = (
                2.0
                * (x_j @ w - y_j)
                * x_j
            )

            gradient_memory[j] = (
                new_gradient
            )

            average_gradient += (
                new_gradient
                - old_gradient
            ) / n

            gradient = (
                average_gradient
            )


        else:

            raise ValueError(
                f"Неизвестный метод: {method}"
            )

        eta = get_eta(
            base_step=base_step,
            k=k,
            lr_mode=lr_mode
        )

        if method in [
            "VGD",
            "SGD",
            "SAG"
        ]:

            w = (
                w
                - eta * gradient
            )


        elif method == "Momentum":

            h = (
                alpha * h
                + eta * gradient
            )

            w = (
                w - h
            )


        elif method == "Adam":

            m = (
                beta1 * m
                + (1.0 - beta1)
                * gradient
            )

            v = (
                beta2 * v
                + (1.0 - beta2)
                * gradient ** 2
            )

            t = k + 1

            m_hat = (
                m
                / (1.0 - beta1 ** t)
            )

            v_hat = (
                v
                / (1.0 - beta2 ** t)
            )

            w = (
                w
                - eta
                * m_hat
                / (
                    np.sqrt(v_hat)
                    + epsilon
                )
            )


        if (
            not np.all(np.isfinite(w))
            or np.linalg.norm(w) > 1e8
        ):
            break

        val_loss, _ = evaluate_weights(
            X_val,
            y_val,
            w
        )

        val_history.append(
            val_loss
        )



        if (
            np.isfinite(val_loss)
            and val_loss < best_val_loss
        ):

            best_val_loss = val_loss

            best_weights = (
                w.copy()
            )

            best_iteration = (
                k + 1
            )


    return {
        "weights": best_weights,
        "best_val_loss": best_val_loss,
        "best_iteration": best_iteration,
        "iterations_done": len(
            val_history
        ),
        "val_history": val_history
    }


def investigate(
    method,
    lr_mode,
    batch_size=32,
    alpha=0.9,
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8
):
    """
    Перебирает:
        eta     для constant
        lambda  для TimeDecay

    Лучший параметр выбирается ТОЛЬКО
    по минимальному validation loss.

    Test используется только после выбора.
    """

    rows = []
    runs = []

    parameter_name = (
        "eta"
        if lr_mode == "constant"
        else "lambda"
    )

    for base_step in STEP_GRID:

        run = train_optimizer(
            method=method,
            lr_mode=lr_mode,
            base_step=float(base_step),
            max_iter=MAX_ITER,
            batch_size=batch_size,
            alpha=alpha,
            beta1=beta1,
            beta2=beta2,
            epsilon=epsilon,
            random_state=RANDOM_STATE
        )

        train_loss, train_r2 = (
            evaluate_weights(
                X_train,
                y_train,
                run["weights"]
            )
        )

        rows.append({
            parameter_name:
                float(base_step),

            "Loss train":
                train_loss,

            "R2 train":
                train_r2,

            "Loss val":
                run["best_val_loss"],

            "Best iteration":
                run["best_iteration"]
        })

        runs.append(run)


    results = pd.DataFrame(
        rows
    )

    best_position = int(
        np.argmin(
            results[
                "Loss val"
            ].to_numpy()
        )
    )

    best_step = float(
        results.iloc[
            best_position
        ][parameter_name]
    )

    best_run = runs[
        best_position
    ]


    loss_train, r2_train = (
        evaluate_weights(
            X_train,
            y_train,
            best_run["weights"]
        )
    )


    loss_test, r2_test = (
        evaluate_weights(
            X_test,
            y_test,
            best_run["weights"]
        )
    )


    if lr_mode == "constant":

        step_description = (
            f"eta = {best_step:g}"
        )

    else:

        step_description = (
            f"eta_k = {best_step:g} * "
            f"(1 / (1 + k))^0.5"
        )


    summary = {
        "Метод":
            method,

        "Лучший шаг":
            step_description,

        "Loss train":
            loss_train,

        "Loss test":
            loss_test,

        "R2 train":
            r2_train,

        "R2 test":
            r2_test,

        "Число итераций на test":
            best_run[
                "best_iteration"
            ]
    }


    print(
        "=" * 70
    )

    print(
        f"Метод: {method}"
    )

    print(
        "Шаг:",
        (
            "постоянный eta"
            if lr_mode == "constant"
            else "TimeDecay eta(lambda)"
        )
    )

    print(
        "=" * 70
    )

    display(results)

    print(
        f"\nЛучший {parameter_name}: "
        f"{best_step:g}"
    )

    print(
        "Минимальный Loss_val:",
        f"{best_run['best_val_loss']:.4f}"
    )

    print(
        "Лучшая итерация:",
        best_run[
            "best_iteration"
        ]
    )

    print(
        f"Loss_test: {loss_test:.4f}"
    )

    print(
        f"R2_test: {r2_test:.6f}"
    )


    return (
        results,
        summary,
        best_run
    )


(
    vgd_const_results,
    vgd_const_summary,
    vgd_const_best_run
) = investigate(
    method="VGD",
    lr_mode="constant"
)

Метод: VGD
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,7.700681e+07,-0.265260,7.456060e+07,5000
1,0.00010,2.393730e+07,0.606698,2.316450e+07,5000
2,0.00100,1.814174e+07,0.701922,2.095563e+07,5000
3,0.01000,1.519756e+07,0.750297,1.826680e+07,5000
4,0.10000,1.277249e+07,0.790142,1.668339e+07,3927
5,1.00000,1.092759e+08,-0.795457,1.072123e+08,0



Лучший eta: 0.1
Минимальный Loss_val: 16683391.6043
Лучшая итерация: 3927
Loss_test: 27587568.8567
R2_test: 0.604408


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
(
    vgd_decay_results,
    vgd_decay_summary,
    vgd_decay_best_run
) = investigate(
    method="VGD",
    lr_mode="decay"
)

Метод: VGD
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,1.081098e+08,-0.776297,1.060315e+08,5000
1,0.00010,9.839690e+07,-0.616710,9.619739e+07,5000
2,0.00100,4.823216e+07,0.207521,4.569406e+07,5000
3,0.01000,2.019171e+07,0.668240,2.132919e+07,3253
4,0.10000,1.705011e+07,0.719858,1.997754e+07,5000
5,1.00000,1.373641e+07,0.774304,1.705618e+07,5000



Лучший lambda: 1
Минимальный Loss_val: 17056178.2968
Лучшая итерация: 5000
Loss_test: 26457141.6527
R2_test: 0.620618


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
(
    sgd_const_results,
    sgd_const_summary,
    sgd_const_best_run
) = investigate(
    method="SGD",
    lr_mode="constant",
    batch_size=32
)

Метод: SGD
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,7.696110e+07,-0.264509,7.451634e+07,5000
1,0.00010,2.390890e+07,0.607165,2.314957e+07,5000
2,0.00100,1.823504e+07,0.700389,2.078644e+07,4468
3,0.01000,1.550810e+07,0.745194,1.743131e+07,4468
4,0.10000,1.411621e+07,0.768064,1.411865e+07,3681
5,1.00000,1.092759e+08,-0.795457,1.072123e+08,0



Лучший eta: 0.1
Минимальный Loss_val: 14118645.7790
Лучшая итерация: 3681
Loss_test: 31917272.1777
R2_test: 0.542323


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
(
    sgd_decay_results,
    sgd_decay_summary,
    sgd_decay_best_run
) = investigate(
    method="SGD",
    lr_mode="decay",
    batch_size=32
)

Метод: SGD
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,1.080992e+08,-0.776123,1.060211e+08,5000
1,0.00010,9.830241e+07,-0.615158,9.610515e+07,5000
2,0.00100,4.790026e+07,0.212975,4.539456e+07,5000
3,0.01000,1.981025e+07,0.674508,2.132929e+07,4468
4,0.10000,1.711003e+07,0.718874,1.973931e+07,4468
5,1.00000,1.092759e+08,-0.795457,1.072123e+08,0



Лучший lambda: 0.1
Минимальный Loss_val: 19739309.5493
Лучшая итерация: 4468
Loss_test: 24922462.0566
R2_test: 0.642625


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
(
    sag_const_results,
    sag_const_summary,
    sag_const_best_run
) = investigate(
    method="SAG",
    lr_mode="constant"
)

Метод: SAG
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,8.069337e+07,-0.325832,7.826814e+07,5000
1,0.00010,2.396031e+07,0.606320,2.297702e+07,5000
2,0.00100,1.847347e+07,0.696472,2.124754e+07,4779
3,0.01000,1.616959e+07,0.734326,1.734194e+07,4805
4,0.10000,3.185309e+07,0.476638,2.898824e+07,59
5,1.00000,3.271008e+07,0.462557,3.069281e+07,19



Лучший eta: 0.01
Минимальный Loss_val: 17341939.9643
Лучшая итерация: 4805
Loss_test: 26596366.6206
R2_test: 0.618622


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
(
    sag_decay_results,
    sag_decay_summary,
    sag_decay_best_run
) = investigate(
    method="SAG",
    lr_mode="decay"
)

Метод: SAG
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,1.085247e+08,-0.783115,1.064516e+08,5000
1,0.00010,1.020579e+08,-0.676863,9.990258e+07,5000
2,0.00100,5.943631e+07,0.023432,5.679836e+07,5000
3,0.01000,2.085785e+07,0.657295,2.148991e+07,3616
4,0.10000,1.763611e+07,0.710230,2.019104e+07,5000
5,1.00000,2.102542e+07,0.654542,2.062082e+07,4359



Лучший lambda: 0.1
Минимальный Loss_val: 20191040.6120
Лучшая итерация: 5000
Loss_test: 25314804.4218
R2_test: 0.636999


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
(
    momentum_const_results,
    momentum_const_summary,
    momentum_const_best_run
) = investigate(
    method="Momentum",
    lr_mode="constant",
    alpha=0.9
)

Метод: Momentum
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,2.393249e+07,0.606777,2.315924e+07,5000
1,0.00010,1.814251e+07,0.701910,2.095774e+07,5000
2,0.00100,1.519888e+07,0.750275,1.826773e+07,5000
3,0.01000,1.277605e+07,0.790083,1.668313e+07,3907
4,0.10000,1.281420e+07,0.789456,1.668003e+07,370
5,1.00000,1.092759e+08,-0.795457,1.072123e+08,0



Лучший eta: 0.1
Минимальный Loss_val: 16680031.2600
Лучшая итерация: 370
Loss_test: 27545063.4689
R2_test: 0.605018


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
(
    momentum_decay_results,
    momentum_decay_summary,
    momentum_decay_best_run
) = investigate(
    method="Momentum",
    lr_mode="decay",
    alpha=0.9
)

Метод: Momentum
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,9.840340e+07,-0.616817,9.620396e+07,5000
1,0.00010,4.819622e+07,0.208112,4.565661e+07,5000
2,0.00100,2.021143e+07,0.667916,2.133440e+07,3166
3,0.01000,1.704989e+07,0.719862,1.997722e+07,5000
4,0.10000,1.373458e+07,0.774334,1.705300e+07,5000
5,1.00000,1.452253e+07,0.761388,1.653100e+07,66



Лучший lambda: 1
Минимальный Loss_val: 16530998.0011
Лучшая итерация: 66
Loss_test: 29072888.6152
R2_test: 0.583110


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [14]:
(
    adam_const_results,
    adam_const_summary,
    adam_const_best_run
) = investigate(
    method="Adam",
    lr_mode="constant",
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8
)

Метод: Adam
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,1.092698e+08,-0.795357,1.072061e+08,5000
1,0.00010,1.092149e+08,-0.794456,1.071506e+08,5000
2,0.00100,1.086679e+08,-0.785468,1.065973e+08,5000
3,0.01000,1.033469e+08,-0.698042,1.012122e+08,5000
4,0.10000,6.366215e+07,-0.046001,6.090486e+07,5000
5,1.00000,1.440625e+07,0.763298,1.728593e+07,5000



Лучший eta: 1
Минимальный Loss_val: 17285930.8601
Лучшая итерация: 5000
Loss_test: 25657350.9150
R2_test: 0.632087


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [15]:
(
    adam_decay_results,
    adam_decay_summary,
    adam_decay_best_run
) = investigate(
    method="Adam",
    lr_mode="decay",
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8
)

Метод: Adam
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,1.092757e+08,-0.795454,1.072121e+08,5000
1,0.00010,1.092742e+08,-0.795429,1.072106e+08,5000
2,0.00100,1.092588e+08,-0.795177,1.071950e+08,5000
3,0.01000,1.091053e+08,-0.792655,1.070398e+08,5000
4,0.10000,1.075818e+08,-0.767622,1.054985e+08,5000
5,1.00000,9.343914e+07,-0.535252,9.117345e+07,5000



Лучший lambda: 1
Минимальный Loss_val: 91173446.8607
Лучшая итерация: 5000
Loss_test: 112636766.8039
R2_test: -0.615154


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [17]:
comparison = pd.DataFrame([
    vgd_const_summary,
    vgd_decay_summary,

    sgd_const_summary,
    sgd_decay_summary,

    sag_const_summary,
    sag_decay_summary,

    momentum_const_summary,
    momentum_decay_summary,

    adam_const_summary,
    adam_decay_summary
])

comparison

,Метод,Лучший шаг,Loss train,Loss test,R2 train,R2 test,Число итераций на test
0,VGD,eta = 0.1,1.277249e+07,2.758757e+07,0.790142,0.604408,3927
1,VGD,eta_k = 1 * (1 / (1 + k))^0.5,1.373641e+07,2.645714e+07,0.774304,0.620618,5000
2,SGD,eta = 0.1,1.411621e+07,3.191727e+07,0.768064,0.542323,3681
3,SGD,eta_k = 0.1 * (1 / (1 + k))^0.5,1.711003e+07,2.492246e+07,0.718874,0.642625,4468
4,SAG,eta = 0.01,1.616959e+07,2.659637e+07,0.734326,0.618622,4805
5,SAG,eta_k = 0.1 * (1 / (1 + k))^0.5,1.763611e+07,2.531480e+07,0.710230,0.636999,5000
6,Momentum,eta = 0.1,1.281420e+07,2.754506e+07,0.789456,0.605018,370
7,Momentum,eta_k = 1 * (1 / (1 + k))^0.5,1.452253e+07,2.907289e+07,0.761388,0.583110,66
8,Adam,eta = 1,1.440625e+07,2.565735e+07,0.763298,0.632087,5000
9,Adam,eta_k = 1 * (1 / (1 + k))^0.5,9.343914e+07,1.126368e+08,-0.535252,-0.615154,5000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

In [2]:
#1) По данной таблице я сделала вывод, что наилучшим вариантом будет SGD c уменьшающимся шагом. Такой вывод я сделала, потому что смотрела на значение Loss train в таблице и поняла, что низкие значения у SGD, Adam и Momentum. Далее стала смотреть на Loss test, так как то, что  у нас хороший результат на train не значит что на test он тоже будет такой. Тут loss уже больше. но хорошее значение у SGD c уменьшающимся шагом. У остальных значения уже выше, но не будем делать поспешных выводов. Также проанализируем столбики с R. Тут уже чем ближе к 1, тем лучше. Тут тоже самое высокое значение у SGD c уменьшающимся шагом. Таким образом я и выбрала SGD с уменьшающимся шаком. Еще добавлю что последняя колонка показывает на какой итерации была получена минимальная ошибка, но по этому параметру нельзя сказать, что самое меньшее значение автоматически самая лучшая моделька, так как у модельки с наименьшим значением "Число итераций на test" (использовали Adam + уменьш шаг), но результаты не самые лучшие по другим колонкам.

In [ ]:
#2) Сначала думаю стоит объяснить просто значение R^2, Я понимаю это как коэффициент, который говорит нам насколько наше значение модели лучше чем просто обычное среднее значение. По сути сравниваем с очень простой моделью, которая ничего не знает об автомобилях и просто всегда ставит ср. цену. Если R^2=1, то это идеальное попадание в реальные значения, если R^2=0, то это ничем не лучше чем среднее, если R^2<1, то еще хуже чем среднее. В нашей таблице разница в том, на каких данных посчитана R^2 - на train или test. Важно смотреть на разницу между этими двумя значениями. Оба высокие - круто, модель хорошо предсказывает как на трейн так и на тест. у нас для SGD (не фиксированный шаг) значение на тест ниже, что в целом имеет место быть. Может тогда сильно подстроиться под обучающие данные или выборки заметно отличаются.

In [ ]:
#R^2 test как раз показывает, сохранилось ли качество модели на данных, которые она не использовала при обучении (по сути не видела их). Чем выше значение, тем лучше метод работает на новых данных. Обязательно нужно смотреть на разницу между трейн и тест, если она небольшая, значит качество модели переносится на новые хорошо, если разница большая, значит модель намного лучше работает на обучающих данных, чем на новых. Возможно переобучение